# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`

This notebook provides an example workflow for loading and exploring the ["A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya"](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and accessed programmatically below.

In [ ]:
# Ensure `mlcroissant` is installed--run once per environment
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the URL for this FAIR² Mental Health Survey Croissant schema
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

# Optionally, show citation and keywords
print(f"Citation: {getattr(metadata, 'cite_as', getattr(metadata, 'citeAs', None))}")
print(f"Keywords: {getattr(metadata, 'keywords', None)}")

## 2. Data Overview

Review available record sets and their field and column IDs. Croissant datasets structure their data into one or more *record sets* (similar to tables in a relational database). Each record set contains fields (columns) that can be referenced and extracted.

Below, we list all record sets and their associated fields and columns by their `@id` (unique Croissant identifier).

In [ ]:
# List all available record sets and details
print("Available record sets in this dataset:\n")
rs_list = list(dataset.record_sets)

for rs in rs_list:
    print(f"Record Set: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Fields and Columns with @id:")
    for field in rs.fields:
        print(f"   - Field: {getattr(field, 'id', None)}, Title: {getattr(field, 'name', None)}")
        if hasattr(field, 'columns'):
            for col in field.columns:
                print(f"      * Column: {col.id}, Title: {col.name}")
    print()

In [ ]:
# Optionally, peek at a record from each set
for rs in rs_list:
    print(f"Sample record from record set {rs.id}:")
    records = dataset.records(record_set=rs.id)
    try:
        first_rec = next(records)
        print(first_rec)  # prints a dict with field/column @id as keys
    except Exception as e:
        print(f"   (No records or error: {e})")
    print()

## 3. Data Extraction

Load data from a specific record set (table) into a Pandas DataFrame for analysis. All IDs must be referenced by their `@id` fields as listed above.

We will extract all record sets into separate DataFrames for flexible exploration.

In [ ]:
# Extract all record sets into DataFrames
record_sets = [rs.id for rs in rs_list]
dataframes = {}

for record_set_id in record_sets:
    # Safely convert records to DataFrame
    records_gen = dataset.records(record_set=record_set_id)
    records_list = list(records_gen)
    df = pd.DataFrame(records_list)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for Record Set: {record_set_id} → {df.shape[0]} rows, {df.shape[1]} columns.")

# Pick the main survey record set by its Croissant @id (update as appropriate for the dataset)
main_record_set_id = record_sets[0] if record_sets else None
print(f"\nFields (@id) in main record set '{main_record_set_id}':\n", dataframes[main_record_set_id].columns.tolist())

# Show some data
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Now, let's apply common data processing steps to the main survey record set. We'll filter records based on a numeric field, normalize values, and group by a categorical variable for further insights.

First, inspect the available numeric and groupable fields (by `@id`).

In [ ]:
# Choose a numeric field by its `@id` (select one such as GAD-7 score if available)
numeric_candidates = [col for col in dataframes[main_record_set_id].columns if dataframes[main_record_set_id][col].dtype in ['int64', 'float64', 'int32', 'float32', 'int16', 'float16']]
print("Numeric field candidates for analysis (@id):", numeric_candidates)

# Manually set (or pick) a numeric field for demo; update this to match your dataset's GAD-7/PHQ-9/etc. field @id
numeric_field = numeric_candidates[0] if numeric_candidates else None
print(f"Using numeric field for analysis: {numeric_field}")

# Choose a group field (e.g., gender @id or education level @id), update as necessary:
group_candidates = [col for col in dataframes[main_record_set_id].columns if dataframes[main_record_set_id][col].dtype == 'object']
print("Categorical/grouping field candidates for analysis (@id):", group_candidates)
group_field = group_candidates[0] if group_candidates else None
print(f"Using group field: {group_field}")

In [ ]:
# --- Data analysis steps with the chosen fields ---
# Remove outliers: filter for non-null, plausible values > threshold
threshold = 5
df_main = dataframes[main_record_set_id]
if numeric_field is not None:
    filtered_df = df_main[df_main[numeric_field].apply(lambda x: pd.notnull(x) and x > threshold)]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize selected numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized column '{numeric_field}':")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by group_field
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean {numeric_field} by '{group_field}':")
        print(grouped_df.head())
    else:
        print("No valid grouping field specified.")
else:
    print("No numeric field available for analysis in main record set.")

## 5. Visualization

Visualize the distributions or relationships in the main record set, using the selected numeric and group fields. Common visualizations include histograms, boxplots, and grouped barplots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if numeric_field and filtered_df.shape[0] > 0:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group
    if group_field and group_field in filtered_df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=30, ha="right")
        plt.show()
else:
    print("Insufficient numeric data or columns for visualization.")

## 6. Conclusion

In this notebook, we demonstrated how to load metadata and records from a Croissant-based dataset using the `mlcroissant` library, explored record sets and fields via their `@id`, extracted data for analysis, performed EDA including outlier removal and normalization, grouped by categorical attributes, and created visualizations using the DataFrame. 

All dataset entities (record sets, fields, columns) are referenced strictly by their Croissant `@id` to ensure clarity and reproducibility.

For deeper research, repeat this process on other record sets and fields, or link them as appropriate for your use case.